# OmniVoice Vietnamese Batch TTS — fixed `language_id=vi`

Notebook này tạo TTS tiếng Việt theo từng scene trong JSON.

Output:
- `audio/S001.wav`, `audio/S002.wav`, ...
- `text/S001.txt`, `text/S002.txt`, ...
- `tts_manifest.csv`
- `<job_id>_tts_package.zip`

Điểm sửa chính:
- Không nhét `Vietnamese`, `clear`, `warm`, `natural livestream host voice` vào `instruct` nữa.
- Dùng `LANGUAGE_ID = "vi"` để ép ngôn ngữ tiếng Việt.
- `instruct` chỉ giữ các item hợp lệ như `female`, `young adult`, `moderate pitch`.


In [ ]:
# 1) Install OmniVoice
# Khuyến nghị chạy Colab GPU: Runtime > Change runtime type > T4 GPU
!pip install -q omnivoice soundfile pandas tqdm

In [ ]:
# 2) Config

# IMPORTANT:
# Vietnamese support nằm ở language_id, không nằm trong instruct.
LANGUAGE_ID = "vi"

# Chọn mode:
# - "auto": không truyền instruct, ít lỗi nhất, model tự chọn giọng
# - "design": truyền voice attributes hợp lệ + language_id="vi"
# - "clone": upload/ref audio 3-10s để giữ giọng nhất quán hơn giữa các scene
VOICE_MODE = "auto"  # "auto" | "design" | "clone"

# Chỉ dùng khi VOICE_MODE = "design".
# OmniVoice chỉ nhận các item trong allow-list, phân tách bằng comma + space.
# Không dùng: adult, clear, warm, natural Vietnamese livestream host voice
SAFE_INSTRUCT = "female, young adult, moderate pitch"

# Chỉ dùng khi VOICE_MODE = "clone".
REF_AUDIO_PATH = None  # ví dụ: "/content/ref.wav"; để None thì notebook sẽ hỏi upload
REF_TEXT = None        # optional; nếu None, OmniVoice có thể dùng ASR nếu load_asr=True

# JSON source:
# - UPLOAD_JSON = False: dùng JSON nhúng sẵn trong notebook
# - UPLOAD_JSON = True: upload file JSON mới từ máy
UPLOAD_JSON = False
JSON_PATH = "/content/master_script.json"

OUTPUT_ROOT = "/content/tts_output"
SAMPLE_RATE = 24000
MODEL_ID = "k2-fsa/OmniVoice"

# Test nhanh / resume
MAX_SCENES = None          # ví dụ: 3 để test 3 scene đầu; None = chạy hết
START_FROM_SCENE_ID = None # ví dụ: "S010" để chạy từ S010 trở đi

# Optional generation params
NUM_STEP = 32     # 16 nhanh hơn, 32 ổn định hơn
SPEED = 1.0       # >1 nhanh hơn, <1 chậm hơn
DURATION = None   # ví dụ 5.0 để ép audio 5 giây; None = để model tự sinh tự nhiên

In [ ]:
# 3) Imports

import json
import os
import re
import shutil
import traceback
import zipfile
from pathlib import Path
from datetime import datetime

import pandas as pd
import soundfile as sf
import torch
from tqdm.auto import tqdm
from IPython.display import Audio, display
from google.colab import files

In [ ]:
# 4) Load JSON

EMBEDDED_MASTER_SCRIPT = r'''{
  "job_id": "cocoon_test_001",
  "base_visual_lock": "Cocoon là thương hiệu mỹ phẩm thuần chay Việt Nam, sử dụng nguyên liệu thiên nhiên từ các vùng nông sản nổi tiếng Việt Nam. Hôm nay livestream giới thiệu bộ sản phẩm chăm sóc da và tóc: Tẩy da chết cà phê Đắk Lắk, Tẩy da chết đường thốt nốt An Giang, Nước dưỡng tóc tinh dầu bưởi, và Dầu gội bưởi không sulfate. Tất cả sản phẩm đều lành tính, không thử nghiệm trên động vật, đạt chuẩn CGMP.",
  "global_rules": {
    "aspect_ratio": "9:16",
    "camera": "medium close-up, fixed camera, direct front view",
    "motion_level": "low",
    "scene_duration_sec": 5,
    "transition": "0.2s crossfade",
    "consistency_rule": "All scenes must use the same face, hairstyle, outfit, background, lighting, camera angle, and body framing."
  },
  "playlist": [
    {
      "clip_id": "A_MAIN_SALES_LOOP",
      "purpose": "Loop chính giới thiệu sản phẩm",
      "scenes": [
        "S001",
        "S002",
        "S003",
        "S004",
        "S005",
        "S006",
        "S007",
        "S008",
        "S009",
        "S010",
        "S011",
        "S012",
        "S013",
        "S014"
      ]
    },
    {
      "clip_id": "B_COMMENT_READING_LOOP",
      "purpose": "Loop đọc comment",
      "scenes": [
        "S015",
        "S016",
        "S017",
        "S018"
      ]
    },
    {
      "clip_id": "C_CTA_LOOP",
      "purpose": "Loop chốt đơn",
      "scenes": [
        "S019"
      ]
    }
  ],
  "scenes": [
    {
      "scene_id": "S001",
      "clip_id": "A_MAIN_SALES_LOOP",
      "order": 1,
      "scene_type": "HOST_TALK",
      "duration_target_sec": 5,
      "voiceover": "Dạ em xin chào tất cả anh chị và các bạn đang xem live của Cocoon hôm nay nha! Hôm nay bên em có rất nhiều deal giảm giá độc quyền và quà tặng kèm cực hời chỉ có trên phiên live này thôi ạ.",
      "visual_goal": "Host mở đầu livestream, nhìn thẳng camera, mỉm cười nhẹ.",
      "overlay_text": null,
      "start_anchor": "Host facing camera, hands relaxed near product table.",
      "end_anchor": "Host facing camera, same pose, small smile.",
      "needs_lipsync": true,
      "needs_product_overlay": false
    },
    {
      "scene_id": "S002",
      "clip_id": "A_MAIN_SALES_LOOP",
      "order": 2,
      "scene_type": "HOST_TALK",
      "duration_target_sec": 5,
      "voiceover": "Cocoon là mỹ phẩm thuần chay Việt Nam, lành tính và không thử nghiệm trên động vật.",
      "visual_goal": "Host giới thiệu triết lý thương hiệu, giữ ánh mắt tự nhiên.",
      "overlay_text": "Thuần chay (Vegan), Không thử nghiệm trên động vật (Cruelty-Free), Nguyên liệu thiên nhiên Việt Nam",
      "start_anchor": "Host facing camera, product visible on table.",
      "end_anchor": "Host facing camera, product still visible.",
      "needs_lipsync": true,
      "needs_product_overlay": true
    },
    {
      "scene_id": "S003",
      "clip_id": "A_MAIN_SALES_LOOP",
      "order": 3,
      "scene_type": "PRODUCT_CLOSEUP",
      "duration_target_sec": 5,
      "voiceover": "Hạt cà phê nguyên chất từ Đắk Lắk kết hợp với bơ ca cao Tiền Giang giúp làm sạch da chết cơ thể hiệu",
      "visual_goal": "Cận cảnh sản phẩm Tẩy Da Chết Cà Phê Đắk Lắk trên bàn livestream.",
      "overlay_text": "Tẩy Da Chết Cà Phê Đắk Lắk",
      "start_anchor": "Product centered on table, clean background.",
      "end_anchor": "Product centered, same angle, slight camera push-in.",
      "needs_lipsync": false,
      "needs_product_overlay": true
    },
    {
      "scene_id": "S004",
      "clip_id": "A_MAIN_SALES_LOOP",
      "order": 4,
      "scene_type": "HOST_TALK",
      "duration_target_sec": 5,
      "voiceover": "Loại bỏ tế bào chết hiệu quả",
      "visual_goal": "Host giới thiệu công dụng Tẩy Da Chết Cà Phê Đắk Lắk, cầm sản phẩm nhẹ nhàng.",
      "overlay_text": null,
      "start_anchor": "Host holding product, facing camera.",
      "end_anchor": "Host holding product, same pose.",
      "needs_lipsync": true,
      "needs_product_overlay": true
    },
    {
      "scene_id": "S005",
      "clip_id": "A_MAIN_SALES_LOOP",
      "order": 5,
      "scene_type": "CTA",
      "duration_target_sec": 5,
      "voiceover": "Giảm 15% trên livestream, tặng sample khi mua 2 hũ trở lên",
      "visual_goal": "Host chỉ vào giá sản phẩm Tẩy Da Chết Cà Phê Đắk Lắk trên bàn.",
      "overlay_text": "Tẩy Da Chết Cà Phê Đắk Lắk - Giá live: 115,000đ",
      "start_anchor": "Host facing camera, product visible.",
      "end_anchor": "Host facing camera, product visible.",
      "needs_lipsync": true,
      "needs_product_overlay": true
    },
    {
      "scene_id": "S006",
      "clip_id": "A_MAIN_SALES_LOOP",
      "order": 6,
      "scene_type": "PRODUCT_CLOSEUP",
      "duration_target_sec": 5,
      "voiceover": "Tinh thể đường thốt nốt nhuyễn mịn kết hợp với nam châm dưỡng ẩm Pentavitin, phức hợp dầu chưng cất ",
      "visual_goal": "Cận cảnh sản phẩm Tẩy Da Chết Đường Thốt Nốt An Giang trên bàn livestream.",
      "overlay_text": "Tẩy Da Chết Đường Thốt Nốt An Giang",
      "start_anchor": "Product centered on table, clean background.",
      "end_anchor": "Product centered, same angle, slight camera push-in.",
      "needs_lipsync": false,
      "needs_product_overlay": true
    },
    {
      "scene_id": "S007",
      "clip_id": "A_MAIN_SALES_LOOP",
      "order": 7,
      "scene_type": "HOST_TALK",
      "duration_target_sec": 5,
      "voiceover": "Tẩy tế bào chết siêu nhẹ nhàng, không gây đau rát",
      "visual_goal": "Host giới thiệu công dụng Tẩy Da Chết Đường Thốt Nốt An Giang, cầm sản phẩm nhẹ nhàng.",
      "overlay_text": null,
      "start_anchor": "Host holding product, facing camera.",
      "end_anchor": "Host holding product, same pose.",
      "needs_lipsync": true,
      "needs_product_overlay": true
    },
    {
      "scene_id": "S008",
      "clip_id": "A_MAIN_SALES_LOOP",
      "order": 8,
      "scene_type": "CTA",
      "duration_target_sec": 5,
      "voiceover": "Giảm 15% trên livestream, tặng túi đựng mỹ phẩm khi mua combo",
      "visual_goal": "Host chỉ vào giá sản phẩm Tẩy Da Chết Đường Thốt Nốt An Giang trên bàn.",
      "overlay_text": "Tẩy Da Chết Đường Thốt Nốt An Giang - Giá live: 138,000đ",
      "start_anchor": "Host facing camera, product visible.",
      "end_anchor": "Host facing camera, product visible.",
      "needs_lipsync": true,
      "needs_product_overlay": true
    },
    {
      "scene_id": "S009",
      "clip_id": "A_MAIN_SALES_LOOP",
      "order": 9,
      "scene_type": "PRODUCT_CLOSEUP",
      "duration_target_sec": 5,
      "voiceover": "Sản phẩm treatment dành cho tóc, phù hợp với tình trạng tóc rụng, tóc yếu, tóc thưa mỏng. Ngăn ngừa ",
      "visual_goal": "Cận cảnh sản phẩm Nước Dưỡng Tóc Tinh Dầu Bưởi trên bàn livestream.",
      "overlay_text": "Nước Dưỡng Tóc Tinh Dầu Bưởi",
      "start_anchor": "Product centered on table, clean background.",
      "end_anchor": "Product centered, same angle, slight camera push-in.",
      "needs_lipsync": false,
      "needs_product_overlay": true
    },
    {
      "scene_id": "S010",
      "clip_id": "A_MAIN_SALES_LOOP",
      "order": 10,
      "scene_type": "HOST_TALK",
      "duration_target_sec": 5,
      "voiceover": "Ngăn ngừa 60% nguyên nhân rụng tóc",
      "visual_goal": "Host giới thiệu công dụng Nước Dưỡng Tóc Tinh Dầu Bưởi, cầm sản phẩm nhẹ nhàng.",
      "overlay_text": null,
      "start_anchor": "Host holding product, facing camera.",
      "end_anchor": "Host holding product, same pose.",
      "needs_lipsync": true,
      "needs_product_overlay": true
    },
    {
      "scene_id": "S011",
      "clip_id": "A_MAIN_SALES_LOOP",
      "order": 11,
      "scene_type": "CTA",
      "duration_target_sec": 5,
      "voiceover": "Giảm 14% trên livestream, mua kèm dầu gội tiết kiệm thêm 58k",
      "visual_goal": "Host chỉ vào giá sản phẩm Nước Dưỡng Tóc Tinh Dầu Bưởi trên bàn.",
      "overlay_text": "Nước Dưỡng Tóc Tinh Dầu Bưởi - Giá live: 275,000đ",
      "start_anchor": "Host facing camera, product visible.",
      "end_anchor": "Host facing camera, product visible.",
      "needs_lipsync": true,
      "needs_product_overlay": true
    },
    {
      "scene_id": "S012",
      "clip_id": "A_MAIN_SALES_LOOP",
      "order": 12,
      "scene_type": "PRODUCT_CLOSEUP",
      "duration_target_sec": 5,
      "voiceover": "Từ tinh dầu vỏ bưởi Việt Nam truyền thống kết hợp với vitamin B5, hoạt chất dưỡng ẩm Xylishine™ cùng",
      "visual_goal": "Cận cảnh sản phẩm Dầu Gội Bưởi Không Sulfate trên bàn livestream.",
      "overlay_text": "Dầu Gội Bưởi Không Sulfate",
      "start_anchor": "Product centered on table, clean background.",
      "end_anchor": "Product centered, same angle, slight camera push-in.",
      "needs_lipsync": false,
      "needs_product_overlay": true
    },
    {
      "scene_id": "S013",
      "clip_id": "A_MAIN_SALES_LOOP",
      "order": 13,
      "scene_type": "HOST_TALK",
      "duration_target_sec": 5,
      "voiceover": "Làm sạch tóc và da đầu cực kỳ dịu nhẹ",
      "visual_goal": "Host giới thiệu công dụng Dầu Gội Bưởi Không Sulfate, cầm sản phẩm nhẹ nhàng.",
      "overlay_text": null,
      "start_anchor": "Host holding product, facing camera.",
      "end_anchor": "Host holding product, same pose.",
      "needs_lipsync": true,
      "needs_product_overlay": true
    },
    {
      "scene_id": "S014",
      "clip_id": "A_MAIN_SALES_LOOP",
      "order": 14,
      "scene_type": "CTA",
      "duration_target_sec": 5,
      "voiceover": "Giảm 15% trên livestream, combo với nước dưỡng tóc siêu tiết kiệm",
      "visual_goal": "Host chỉ vào giá sản phẩm Dầu Gội Bưởi Không Sulfate trên bàn.",
      "overlay_text": "Dầu Gội Bưởi Không Sulfate - Giá live: 330,000đ",
      "start_anchor": "Host facing camera, product visible.",
      "end_anchor": "Host facing camera, product visible.",
      "needs_lipsync": true,
      "needs_product_overlay": true
    },
    {
      "scene_id": "S015",
      "clip_id": "B_COMMENT_READING_LOOP",
      "order": 15,
      "scene_type": "HOST_PHONE_READING",
      "duration_target_sec": 5,
      "voiceover": "Dạ câu hỏi này em trả lời ngay cho mình nha.",
      "visual_goal": "Host nhìn xuống điện thoại như đang đọc comment livestream.",
      "overlay_text": "Đang trả lời comment...",
      "start_anchor": "Host holding phone, looking slightly down.",
      "end_anchor": "Host still holding phone, looking slightly down.",
      "needs_lipsync": true,
      "needs_product_overlay": false
    },
    {
      "scene_id": "S016",
      "clip_id": "B_COMMENT_READING_LOOP",
      "order": 16,
      "scene_type": "FAQ_ANSWER",
      "duration_target_sec": 5,
      "voiceover": "Dạ đây là sản phẩm chính hãng 100% từ Cocoon Vietnam, có đầy đủ tem nhãn và chứng nhận. Mọi người yên tâm ạ.",
      "visual_goal": "Host trả lời câu hỏi, nhìn camera tự nhiên.",
      "overlay_text": "Sản phẩm có phải hàng chính hãng không?",
      "start_anchor": "Host facing camera, phone in hand.",
      "end_anchor": "Host facing camera, nodding gently.",
      "needs_lipsync": true,
      "needs_product_overlay": false
    },
    {
      "scene_id": "S017",
      "clip_id": "B_COMMENT_READING_LOOP",
      "order": 17,
      "scene_type": "FAQ_ANSWER",
      "duration_target_sec": 5,
      "voiceover": "Sản phẩm tẩy da chết cà phê và đường thốt nốt được thiết kế cho cơ thể, không khuyến nghị dùng cho vùng da mặt vì hạt tẩ",
      "visual_goal": "Host trả lời câu hỏi, nhìn camera tự nhiên.",
      "overlay_text": "Tẩy da chết có dùng được cho mặt không?",
      "start_anchor": "Host facing camera, phone in hand.",
      "end_anchor": "Host facing camera, nodding gently.",
      "needs_lipsync": true,
      "needs_product_overlay": false
    },
    {
      "scene_id": "S018",
      "clip_id": "B_COMMENT_READING_LOOP",
      "order": 18,
      "scene_type": "FAQ_ANSWER",
      "duration_target_sec": 5,
      "voiceover": "Tẩy da chết cà phê có hạt to hơn, phù hợp da thường đến dầu, tẩy mạnh hơn và có mùi cà phê thơm. Tẩy da chết đường thốt ",
      "visual_goal": "Host trả lời câu hỏi, nhìn camera tự nhiên.",
      "overlay_text": "Hai loại tẩy da chết khác nhau thế nào?",
      "start_anchor": "Host facing camera, phone in hand.",
      "end_anchor": "Host facing camera, nodding gently.",
      "needs_lipsync": true,
      "needs_product_overlay": false
    },
    {
      "scene_id": "S019",
      "clip_id": "C_CTA_LOOP",
      "order": 19,
      "scene_type": "CTA",
      "duration_target_sec": 5,
      "voiceover": "Cả nhà ơi, những ưu đãi này chỉ có trong phiên live hôm nay thôi nha! Ai muốn mua thì bấm vào giỏ hàng góc dưới màn hình hoặc comment số điện thoại để em hỗ trợ lên đơn ngay ạ.",
      "visual_goal": "Host nhìn camera, mỉm cười và chỉ nhẹ xuống góc dưới màn hình.",
      "overlay_text": "Chốt đơn tại giỏ hàng",
      "start_anchor": "Host facing camera, product visible on table.",
      "end_anchor": "Host facing camera, product visible on table.",
      "needs_lipsync": true,
      "needs_product_overlay": true
    }
  ]
}'''

if UPLOAD_JSON:
    print("Upload master_script.json:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No JSON file uploaded.")
    JSON_PATH = next(iter(uploaded.keys()))
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
else:
    data = json.loads(EMBEDDED_MASTER_SCRIPT)
    with open(JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

job_id = data.get("job_id", "tts_job")
scenes = data.get("scenes", [])

if not scenes:
    raise ValueError("JSON không có field scenes hoặc scenes rỗng.")

print("job_id:", job_id)
print("total scenes:", len(scenes))
print("first scene:", scenes[0].get("scene_id"), scenes[0].get("voiceover", "")[:100])

In [ ]:
# 5) Prepare scenes

def safe_filename(name: str) -> str:
    name = str(name or "scene").strip()
    name = re.sub(r"[^a-zA-Z0-9_.-]+", "_", name)
    return name or "scene"

def normalize_text(text: str) -> str:
    text = str(text or "").strip()
    text = re.sub(r"\s+", " ", text)
    return text

prepared = []
start_enabled = START_FROM_SCENE_ID is None

for scene in scenes:
    scene_id = safe_filename(scene.get("scene_id", "scene"))

    if START_FROM_SCENE_ID and scene_id == START_FROM_SCENE_ID:
        start_enabled = True
    if not start_enabled:
        continue

    text = normalize_text(scene.get("voiceover", ""))
    if not text:
        continue

    prepared.append({
        **scene,
        "scene_id": scene_id,
        "voiceover": text,
        "audio_filename": f"{scene_id}.wav",
        "text_filename": f"{scene_id}.txt",
    })

if MAX_SCENES is not None:
    prepared = prepared[:MAX_SCENES]

if not prepared:
    raise ValueError("Không có scene hợp lệ để tạo TTS.")

pd.DataFrame([
    {
        "scene_id": s["scene_id"],
        "audio_filename": s["audio_filename"],
        "duration_target_sec": s.get("duration_target_sec"),
        "text": s["voiceover"],
    }
    for s in prepared
])

In [ ]:
# 6) Validate/sanitize instruct

VALID_ENGLISH_INSTRUCT_ITEMS = {
    "american accent", "australian accent", "british accent", "canadian accent",
    "child", "chinese accent", "elderly", "female", "high pitch",
    "indian accent", "japanese accent", "korean accent", "low pitch",
    "male", "middle-aged", "moderate pitch", "portuguese accent",
    "russian accent", "teenager", "very high pitch", "very low pitch",
    "whisper", "young adult",
}

INSTRUCT_ALIASES = {
    "adult": "young adult",
    "normal pitch": "moderate pitch",
    "medium pitch": "moderate pitch",
}

def sanitize_instruct(instruct: str):
    if not instruct:
        return None, []

    raw_items = [x.strip().lower() for x in instruct.split(",") if x.strip()]
    kept = []
    dropped = []

    for item in raw_items:
        mapped = INSTRUCT_ALIASES.get(item, item)
        if mapped in VALID_ENGLISH_INSTRUCT_ITEMS:
            if mapped not in kept:
                kept.append(mapped)
        else:
            dropped.append(item)

    return ", ".join(kept) if kept else None, dropped

safe_instruct, dropped_instruct = sanitize_instruct(SAFE_INSTRUCT)

print("VOICE_MODE:", VOICE_MODE)
print("LANGUAGE_ID:", LANGUAGE_ID)
print("safe_instruct:", safe_instruct)
if dropped_instruct:
    print("Dropped unsupported instruct items:", dropped_instruct)

if VOICE_MODE == "design" and not safe_instruct:
    print("Không có instruct hợp lệ, sẽ fallback gần giống auto mode.")

In [ ]:
# 7) Load OmniVoice model

from omnivoice import OmniVoice

if torch.cuda.is_available():
    device_map = "cuda:0"
    dtype = torch.float16
else:
    device_map = "cpu"
    dtype = torch.float32

print("device_map:", device_map)
print("dtype:", dtype)

model = OmniVoice.from_pretrained(
    MODEL_ID,
    device_map=device_map,
    dtype=dtype,
    load_asr=True,
)

In [ ]:
# 8) Optional: upload reference audio for clone mode

if VOICE_MODE == "clone" and REF_AUDIO_PATH is None:
    print("Upload reference audio 3-10s, preferably Vietnamese voice, wav/mp3/flac:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Clone mode cần reference audio.")
    REF_AUDIO_PATH = next(iter(uploaded.keys()))

if VOICE_MODE == "clone":
    print("REF_AUDIO_PATH:", REF_AUDIO_PATH)

In [ ]:
# 9) Generate one wav per scene

output_dir = Path(OUTPUT_ROOT) / job_id
audio_dir = output_dir / "audio"
text_dir = output_dir / "text"

audio_dir.mkdir(parents=True, exist_ok=True)
text_dir.mkdir(parents=True, exist_ok=True)

manifest_rows = []
errors = []

def call_omnivoice_generate(text: str):
    kwargs = {
        "text": text,
        "language_id": LANGUAGE_ID,
        "num_step": NUM_STEP,
        "speed": SPEED,
    }

    if DURATION is not None:
        kwargs["duration"] = DURATION

    if VOICE_MODE == "clone":
        kwargs["ref_audio"] = REF_AUDIO_PATH
        if REF_TEXT:
            kwargs["ref_text"] = REF_TEXT

    elif VOICE_MODE == "design":
        if safe_instruct:
            kwargs["instruct"] = safe_instruct

    elif VOICE_MODE == "auto":
        pass
    else:
        raise ValueError('VOICE_MODE phải là "auto", "design", hoặc "clone".')

    try:
        return model.generate(**kwargs)
    except TypeError as e:
        # Backward compatibility fallback nếu bản omnivoice quá cũ không nhận language_id/num_step/speed.
        # Khuyến nghị vẫn nên dùng bản mới nhất.
        msg = str(e)
        fallback_keys = ["language_id", "num_step", "speed", "duration"]
        if any(k in msg for k in fallback_keys) or "unexpected keyword" in msg:
            print("Fallback: model.generate không nhận một số generation kwargs, thử lại với args tối thiểu.")
            minimal_kwargs = {"text": text}
            if VOICE_MODE == "clone":
                minimal_kwargs["ref_audio"] = REF_AUDIO_PATH
                if REF_TEXT:
                    minimal_kwargs["ref_text"] = REF_TEXT
            elif VOICE_MODE == "design" and safe_instruct:
                minimal_kwargs["instruct"] = safe_instruct
            return model.generate(**minimal_kwargs)
        raise

for scene in tqdm(prepared, desc="Generating TTS"):
    scene_id = scene["scene_id"]
    text = scene["voiceover"]
    audio_path = audio_dir / scene["audio_filename"]
    txt_path = text_dir / scene["text_filename"]

    txt_path.write_text(text, encoding="utf-8")

    try:
        audio = call_omnivoice_generate(text)
        wav = audio[0]
        sf.write(audio_path, wav, SAMPLE_RATE)

        manifest_rows.append({
            "scene_id": scene_id,
            "clip_id": scene.get("clip_id"),
            "order": scene.get("order"),
            "scene_type": scene.get("scene_type"),
            "language_id": LANGUAGE_ID,
            "voice_mode": VOICE_MODE,
            "instruct": safe_instruct if VOICE_MODE == "design" else None,
            "audio_filename": str(audio_path),
            "text_filename": str(txt_path),
            "text": text,
            "status": "ok",
        })

        print("OK", scene_id, "->", audio_path.name)

    except Exception as e:
        err = {
            "scene_id": scene_id,
            "audio_filename": str(audio_path),
            "error": repr(e),
            "traceback": traceback.format_exc(),
        }
        errors.append(err)
        manifest_rows.append({
            "scene_id": scene_id,
            "clip_id": scene.get("clip_id"),
            "order": scene.get("order"),
            "scene_type": scene.get("scene_type"),
            "language_id": LANGUAGE_ID,
            "voice_mode": VOICE_MODE,
            "instruct": safe_instruct if VOICE_MODE == "design" else None,
            "audio_filename": str(audio_path),
            "text_filename": str(txt_path),
            "text": text,
            "status": "error",
            "error": repr(e),
        })
        print("ERROR", scene_id, repr(e))

manifest_path = output_dir / "tts_manifest.csv"
errors_path = output_dir / "tts_errors.json"

pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False, encoding="utf-8-sig")
errors_path.write_text(json.dumps(errors, ensure_ascii=False, indent=2), encoding="utf-8")

print("Done.")
print("OK:", sum(1 for r in manifest_rows if r.get("status") == "ok"))
print("Errors:", len(errors))
print("Output dir:", output_dir)

In [ ]:
# 10) Preview first successful audio

ok_rows = [r for r in manifest_rows if r.get("status") == "ok"]
if ok_rows:
    first_audio = ok_rows[0]["audio_filename"]
    print(first_audio)
    display(Audio(first_audio, rate=SAMPLE_RATE))
else:
    print("No successful audio generated. Check tts_errors.json.")

In [ ]:
# 11) Zip and download

zip_path = Path(f"/content/{job_id}_tts_package.zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in output_dir.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=str(path.relative_to(output_dir)))

print("ZIP:", zip_path)
files.download(str(zip_path))